# 03 — Latest completed 5-minute bar → model scores → paper-trading shortlist

Reads the latest timestamp from Gold and loads model URIs from Notebook 02's registry. All tickers receive the ranker score; only **top candidates** are sent through optional triple-barrier meta-model. A market-level GMM regime router applies position-size risk controls. **No trades are placed.** This is an *observable research shortlist* and does not claim to be investment advice or a live execution system.

**Schedule:** existing external feed → completed 5-minute bars Delta → Notebook 01 (`incremental`) → Notebook 03. Wait until full universe for the timestamp arrives; upstream must expose a completeness watermark/bar-closed event. The notebook refuses stale live bars in `live` mode. Use `demo` with generated historical bars.

In [ ]:
import json
from datetime import datetime, timezone
import numpy as np, pandas as pd
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import mlflow.sklearn

spark.conf.set('spark.sql.session.timeZone','UTC')
for name,default in [
    ('catalog','main'),('schema','stock_alpha'),
    ('run_mode','demo'),                    # demo | live
    ('max_staleness_minutes','15'),
    ('top_fraction','0.10'),                # use e.g. 0.01 for 5,000 stocks
    ('meta_threshold','0.55'),
    ('max_names','30'),('per_name_cap','0.01'),('portfolio_cap','0.05'),
    ('min_universe_size','20')
]: dbutils.widgets.text(name,default)
config={k:dbutils.widgets.get(k).strip() for k in ['catalog','schema','run_mode','max_staleness_minutes','top_fraction','meta_threshold','max_names','per_name_cap','portfolio_cap','min_universe_size']}
assert config['run_mode'] in {'demo','live'}
assert 0<float(config['top_fraction'])<=0.5
B=f"`{config['catalog']}`.`{config['schema']}`"
FEATURE_TABLE=f"{B}.`equity_features_5m`"
REGISTRY=f"{B}.`alpha_model_registry`"
SCORE_TABLE=f"{B}.`alpha_paper_signals`"
row=spark.table(REGISTRY).orderBy(F.desc('trained_at')).limit(1).collect()
if not row: raise ValueError('Train Notebook 02 before scoring')
model_record=row[0].asDict()
model_run_id=model_record['run_id']
features=json.loads(model_record['features_json'])
meta_features=json.loads(model_record['meta_features_json'])
regime_features=json.loads(model_record['regime_features_json'])
regime_names={int(k):v for k,v in json.loads(model_record['regime_names_json']).items()}
ranker=mlflow.sklearn.load_model(model_record['ranker_uri'])
meta=mlflow.sklearn.load_model(model_record['meta_uri']) if model_record['meta_uri'] else None
regime=mlflow.sklearn.load_model(model_record['regime_uri'])
print('MLflow run:',model_run_id)

## Score just the newest *complete* timestamp
A five-minute production workflow should use a `bar_complete`/universe coverage flag from ingestion. A maximum timestamp alone **does not prove** that all assets have arrived. The minimum-universe count below catches egregious feed failure; tighten it using the expected active-symbol list and per-symbol lateness SLA.

In [ ]:
gold=spark.table(FEATURE_TABLE)
max_time=gold.agg(F.max('event_time').alias('t')).first()['t']
if max_time is None: raise ValueError('Feature table is empty')
now=datetime.now(timezone.utc).replace(tzinfo=None)
age_minutes=(now-max_time).total_seconds()/60
print('Latest completed-bar end UTC:',max_time,'Age (minutes):',round(age_minutes,1))
if config['run_mode']=='live' and (age_minutes<0 or age_minutes>int(config['max_staleness_minutes'])):
    raise ValueError('Stale or future-dated market bar. No new live paper signals produced.')
cols=['event_time','symbol','sector','close','volume','adv_20']+list(dict.fromkeys(features+[c for c in meta_features if c!='rank_score']+regime_features))
latest=gold.filter(F.col('event_time')==F.lit(max_time)).select(*cols).toPandas()
if len(latest)<int(config['min_universe_size']):
    raise ValueError(f'Universe incomplete: {len(latest)} assets; refusing to score')
for col in set(features+meta_features+regime_features)-{'rank_score'}:
    latest[col]=pd.to_numeric(latest[col],errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0)
latest['rank_score']=ranker.predict(latest[features])
latest.sort_values(['rank_score','symbol'],ascending=[False,True],inplace=True)
latest['rank']=np.arange(1,len(latest)+1)
latest['rank_pct']=1-(latest['rank']-1)/len(latest)
print('Scored universe:',len(latest))

## Route through a GMM market regime and optional meta-model
High-volatility regimes block new buys; medium-volatility regimes reduce size by half. If the meta-model was not trained, the system leaves meta-probability blank and does **not** automatically approve paper buys. Relative rank is within this completed timestamp. Position weights represent **notional allocation caps**, not a trade recommendation; no broker order is sent.

In [ ]:
market_snapshot=latest[regime_features].median(numeric_only=True).to_frame().T
cluster=int(regime.predict(market_snapshot[regime_features])[0])
state=regime_names.get(cluster,'UNKNOWN')
risk_multiplier={'LOW_VOL':1.0,'MID_VOL':0.5,'HIGH_VOL':0.0}.get(state,0.0)
latest['market_regime']=state
latest['risk_multiplier']=risk_multiplier
latest['meta_probability']=np.nan

fraction=float(config['top_fraction'])
top_n=max(1,int(np.ceil(len(latest)*fraction)))
top_n=min(top_n,int(config['max_names']))
indices=latest.head(top_n).index
if meta is not None and len(indices):
    for c in meta_features:
        latest[c]=pd.to_numeric(latest[c],errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0)
    probs=meta.predict_proba(latest.loc[indices,meta_features])[:,list(meta.classes_).index(1)]
    latest.loc[indices,'meta_probability']=probs

approved=(latest.index.isin(indices)&latest['meta_probability'].ge(float(config['meta_threshold']))&
          (risk_multiplier>0))
latest['action']=np.where(approved,'PAPER_BUY_CANDIDATE','WATCH_ONLY')
num_approved=int(approved.sum())
base_weight=min(float(config['per_name_cap']),
                float(config['portfolio_cap'])/num_approved if num_approved else 0.0)
latest['paper_weight']=np.where(approved,base_weight*risk_multiplier,0.0)
latest['model_run_id']=model_run_id
latest['scored_at']=datetime.now(timezone.utc).replace(tzinfo=None)
print('Regime:',state,'Risk multiplier:',risk_multiplier)
print('Paper candidates:',num_approved,'Total paper notional weight:',round(float(latest.paper_weight.sum()),4))
print('These signals are NOT broker orders.')
show_cols=['event_time','symbol','sector','close','adv_20','rank','rank_score','meta_probability','market_regime','action','paper_weight']
display(spark.createDataFrame(latest[show_cols].head(50).fillna({'meta_probability':-1.0})))

## Save model-versioned paper signals to Delta
Keep model run ID and bar-end time on every row. Re-running a job on the same closed bar updates the same signal row (not another order). The produced `WATCH_ONLY` records are included to let you inspect ranking quality and track model drift.

In [ ]:
output_cols=['event_time','symbol','sector','close','adv_20','rank','rank_pct','rank_score',
             'meta_probability','market_regime','risk_multiplier','action','paper_weight','model_run_id','scored_at']
# Keep a floating dtype even if meta is disabled (all NaN), then normalize NaN to SQL NULL.
ready=latest[output_cols].copy().replace([np.inf,-np.inf],np.nan)
ready['meta_probability']=pd.to_numeric(ready['meta_probability'],errors='coerce').astype('float64')
spark_out=spark.createDataFrame(ready)
spark_out=spark_out.withColumn('meta_probability',F.when(F.isnan('meta_probability'),F.lit(None).cast('double')).otherwise(F.col('meta_probability')))
if not spark.catalog.tableExists(SCORE_TABLE.replace('`','')):
    spark_out.write.format('delta').mode('overwrite').saveAsTable(SCORE_TABLE)
else:
    (DeltaTable.forName(spark,SCORE_TABLE).alias('t')
       .merge(spark_out.alias('s'),"t.symbol=s.symbol AND t.event_time=s.event_time AND t.model_run_id=s.model_run_id")
       .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
print('Paper signals saved to:',SCORE_TABLE)
display(spark.table(SCORE_TABLE).filter(F.col('event_time')==F.lit(max_time))
        .orderBy(F.col('rank').asc()).limit(30))

### Production release checklist
- Upstream: define exact symbol universe (include delisted issues), consolidated vs primary-exchange prints, corporate actions, holidays/DST, early closes, bar-completion watermark and late/corrected trades.
- Validation: run event-driven execution simulator, walk-forward retraining across many non-overlapping windows, **fees/spread/slippage** and price gaps, benchmark and sector attribution, out-of-sample calibration, market-impact and capacity analysis. Triple barriers use OHLC, so intra-bar execution ordering is uncertain even with conservative tie handling.
- Operations: run failure alerts and freshness checks, monitor model drift and rank coverage, audit decisions, canary paper trade for weeks/months before considering live trading. Get data licensing and financial/compliance approvals for your jurisdiction.

**Optional future step:** true quote/trade-level order-flow signals or L2 models require event-sequenced bid/ask updates with timestamps, and should be implemented as a *separate* intraday strategy rather than conflated with the bar-level snapshots above.